# Enterprise Industrial AI Copilot

## Notebook 01

# Enterprise Document Discovery & Inventory Pipeline

---

### Author

Charan Teja Arangi

---

### Objective

Build a production-grade enterprise document discovery pipeline capable of scanning an industrial repository, validating documents, generating metadata, creating inventory datasets, and preparing the foundation for OCR, RAG, Knowledge Graph, and Semantic Search pipelines.

---

### Input

data/raw/

---

### Outputs

- document_inventory.csv
- file_hashes.csv
- dataset_statistics.json
- folder_summary.json
- processing_log.txt

In [ ]:
# ==========================================================
# Imports
# ==========================================================

from pathlib import Path
from datetime import datetime
from collections import Counter

import pandas as pd
import hashlib
import mimetypes
import logging
import json
import os

print("Libraries imported successfully.")

In [ ]:
# ==========================================================
# Universal Notebook Setup
# Compatible with VS Code • GitHub • Local Development
# ==========================================================

import sys
from pathlib import Path

# ----------------------------------------------------------
# Step 1: Automatically locate PROJECT_ROOT
# ----------------------------------------------------------
_current_dir = Path.cwd().resolve()
PROJECT_ROOT = None

# Search upwards (max 5 levels) for src/core/config.py
for _ in range(5):
    if (_current_dir / "src" / "core" / "config.py").is_file():
        PROJECT_ROOT = _current_dir
        break
    _current_dir = _current_dir.parent

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "❌ Could not automatically determine PROJECT_ROOT.\n"\
        "Please make sure the notebook is opened from inside the project."
    )

# ----------------------------------------------------------
# Step 2: Add PROJECT_ROOT to Python path (if needed)
# ----------------------------------------------------------
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ----------------------------------------------------------
# Step 3: Import centralized configuration
# ----------------------------------------------------------
from src.core.config import (
    PROJECT_ROOT as CONFIG_ROOT,
    DATA_DIR,
    RAW_DIR,
    PROCESSED_DIR,
    CHUNK_DIR,
    EMBEDDING_DIR,
    VECTOR_DB_DIR,
    KG_DIR,
    INVENTORY_DIR,
    LOG_DIR,
)

# ----------------------------------------------------------
# Step 4: Verify PROJECT_ROOT consistency
# ----------------------------------------------------------
if CONFIG_ROOT != PROJECT_ROOT:
    print("⚠ WARNING: Notebook PROJECT_ROOT differs from config.py PROJECT_ROOT")
    print(f"Notebook : {PROJECT_ROOT}")
    print(f"Config   : {CONFIG_ROOT}")

# Use the centralized PROJECT_ROOT from config.py
PROJECT_ROOT = CONFIG_ROOT

# ----------------------------------------------------------
# Step 5: Verify required directories exist
# ----------------------------------------------------------
required_dirs = [
    DATA_DIR,
    RAW_DIR,
    PROCESSED_DIR,
    CHUNK_DIR,
    EMBEDDING_DIR,
    VECTOR_DB_DIR,
    KG_DIR,
    INVENTORY_DIR,
    LOG_DIR,
]

missing_dirs = [d for d in required_dirs if not d.exists()]

if missing_dirs:
    print("Missing Directories:")
    for d in missing_dirs:
        print(f"   - {d}")
    raise FileNotFoundError("One or more required directories are missing.")

# ----------------------------------------------------------
# Step 6: Environment Verification
# ----------------------------------------------------------
print("=" * 65)
print("PROJECT SETUP VERIFICATION SUMMARY")
print("=" * 65)
print(f"PROJECT_ROOT      : {PROJECT_ROOT}")
print(f"DATA_DIR          : {DATA_DIR}")
print(f"RAW_DIR           : {RAW_DIR}")
print(f"PROCESSED_DIR     : {PROCESSED_DIR}")
print(f"CHUNK_DIR         : {CHUNK_DIR}")
print(f"EMBEDDING_DIR     : {EMBEDDING_DIR}")
print(f"VECTOR_DB_DIR     : {VECTOR_DB_DIR}")
print(f"KG_DIR            : {KG_DIR}")
print(f"INVENTORY_DIR     : {INVENTORY_DIR}")
print(f"LOG_DIR           : {LOG_DIR}")
print("-" * 65)
print("✅ Status          : SUCCESS")
print("=" * 65)


In [ ]:
# ==========================================================
# Create Output Directories
# ==========================================================

INVENTORY_DIR.mkdir(parents=True, exist_ok=True)

LOG_DIR.mkdir(parents=True, exist_ok=True)

print("Directories Ready")

In [ ]:
# ==========================================================
# Logging
# ==========================================================

logging.basicConfig(

    filename=LOG_DIR/"processing_log.txt",

    level=logging.INFO,

    format="%(asctime)s | %(levelname)s | %(message)s",

    force=True

)

logging.info("="*80)
logging.info("Notebook 01 Started")
logging.info("="*80)

print("Logger Ready")

In [ ]:
# ==========================================================
# Validate Folder Structure
# ==========================================================

required = [

    PROJECT_ROOT,

    DATA_DIR,

    RAW_DIR

]

missing = []

for folder in required:

    if not folder.exists():

        missing.append(str(folder))

if len(missing):

    raise FileNotFoundError(

        f"Missing folders:\n{missing}"

    )

print("Project Structure Verified")

In [ ]:
# ==========================================================
# Utility Function
# ==========================================================

def create_document_id(index):

    return f"DOC{index:05d}"

In [ ]:
# ==========================================================
# Utility Function
# ==========================================================

def get_relative_path(path):

    return str(path.relative_to(PROJECT_ROOT))

In [ ]:
# ==========================================================
# Utility Function
# ==========================================================

def bytes_to_mb(size):

    return round(size/(1024*1024),2)

In [ ]:
from src.core.config import SUPPORTED_EXTENSIONS
# ==========================================================
# Notebook Configuration Summary
# ==========================================================

print("="*60)

print("Enterprise Industrial AI Copilot")

print("="*60)

print(f"Project Root : {PROJECT_ROOT}")

print(f"Raw Folder   : {RAW_DIR}")

print(f"Output Folder: {INVENTORY_DIR}")

print(f"Extensions   : {SUPPORTED_EXTENSIONS}")

print("="*60)

In [ ]:
# ==========================================================
# Stage Tracker
# ==========================================================

PIPELINE_STAGES = [

"Document Discovery",

"Validation",

"Metadata Extraction",

"SHA256",

"Duplicate Detection",

"Inventory",

"Statistics",

"Export"

]

for stage in PIPELINE_STAGES:

    print("✓",stage)

In [ ]:
# ==========================================================
# Initial Checks
# ==========================================================

assert RAW_DIR.exists()

assert INVENTORY_DIR.exists()

assert LOG_DIR.exists()

print("Environment Ready")

In [ ]:
# ==========================================================
# Count Raw Files
# ==========================================================

all_files = [

    f

    for f in RAW_DIR.rglob("*")

    if f.is_file()

]

print("Total Raw Files :",len(all_files))

In [ ]:
# ==========================================================
# Preview Raw Files
# ==========================================================

preview = []

for file in all_files[:20]:

    preview.append({

        "File":file.name,

        "Folder":file.parent.name

    })

pd.DataFrame(preview)

In [ ]:
print()

print("="*60)

print("MILESTONE 1 COMPLETED")

print("="*60)

logging.info("Milestone 1 Completed")

# Stage 1 — Enterprise Document Discovery

## Objective

This stage recursively scans the enterprise repository to discover all supported documents.

The discovery engine:

- Searches every subfolder recursively.
- Ignores hidden/system files.
- Filters unsupported file types.
- Creates deterministic Document IDs.
- Produces a clean inventory for downstream processing.

In [ ]:
# ==========================================================
# Recursive Document Discovery
# ==========================================================

def discover_documents(root_dir, supported_extensions):
    """
    Recursively discover supported files.
    """

    discovered = []

    for file in root_dir.rglob("*"):

        if not file.is_file():
            continue

        if file.name.startswith("."):
            continue

        if file.suffix.lower() not in supported_extensions:
            continue

        discovered.append(file)

    discovered = sorted(discovered)

    logging.info(f"Discovered {len(discovered)} supported documents.")

    return discovered

In [ ]:
from src.core.config import SUPPORTED_EXTENSIONS
# ==========================================================
# Execute Discovery
# ==========================================================

documents = discover_documents(
    RAW_DIR,
    SUPPORTED_EXTENSIONS
)

print("Supported Documents Found :", len(documents))

In [ ]:
# ==========================================================
# Build Initial Inventory
# ==========================================================

inventory = []

for idx, file in enumerate(documents, start=1):

    inventory.append({

        "Document_ID": create_document_id(idx),

        "File_Name": file.name,

        "Extension": file.suffix.lower(),

        "Folder": file.parent.name,

        "Relative_Path": get_relative_path(file),

        "Absolute_Path": str(file)

    })

inventory_df = pd.DataFrame(inventory)

inventory_df.head()

In [ ]:
# ==========================================================
# Preview Inventory
# ==========================================================

print("="*70)
print("DOCUMENT INVENTORY PREVIEW")
print("="*70)

display(inventory_df.head(20))

In [ ]:
# ==========================================================
# Discovery Validation
# ==========================================================

assert len(inventory_df) == len(documents)

assert inventory_df["Document_ID"].is_unique

assert inventory_df["File_Name"].notna().all()

assert inventory_df["Relative_Path"].notna().all()

print("Discovery Validation Passed")

In [ ]:
# ==========================================================
# Folder Distribution
# ==========================================================

folder_summary = (

    inventory_df

    .groupby("Folder")

    .size()

    .reset_index(name="Count")

    .sort_values("Count", ascending=False)

)

display(folder_summary)

In [ ]:
# ==========================================================
# Extension Distribution
# ==========================================================

extension_summary = (

    inventory_df

    .groupby("Extension")

    .size()

    .reset_index(name="Count")

    .sort_values("Count", ascending=False)

)

display(extension_summary)

In [ ]:
# ==========================================================
# Quick Statistics
# ==========================================================

print("="*60)

print("Discovery Statistics")

print("="*60)

print(f"Total Documents : {len(inventory_df)}")

print(f"Folders         : {inventory_df['Folder'].nunique()}")

print(f"Extensions      : {inventory_df['Extension'].nunique()}")

print("="*60)

In [ ]:
# ==========================================================
# Check Duplicate Paths
# ==========================================================

duplicate_paths = inventory_df.duplicated(
    subset=["Relative_Path"]
).sum()

print("Duplicate Paths :", duplicate_paths)

In [ ]:
# ==========================================================
# Save Temporary Inventory
# ==========================================================

temp_inventory_path = INVENTORY_DIR / "inventory_stage1.csv"

inventory_df.to_csv(
    temp_inventory_path,
    index=False
)

print("Saved :", temp_inventory_path)

In [ ]:
# ==========================================================
# Stage Completion
# ==========================================================

logging.info("Stage 1 Completed")

print()

print("="*60)

print("MILESTONE 2 COMPLETED")

print("="*60)

# Stage 2 — Enterprise Metadata Extraction

## Objective

This stage enriches every discovered enterprise document with technical metadata.

The generated metadata becomes the foundation for:

- Enterprise Search
- OCR Routing
- Document Analytics
- Knowledge Graph
- Retrieval Augmented Generation
- Compliance Intelligence

Output

Enterprise Metadata Table

In [ ]:
# ==========================================================
# Metadata Extraction
# ==========================================================

def extract_metadata(file_path: Path):

    """
    Extract metadata from a document.

    Returns
    -------
    dict
    """

    stat = file_path.stat()

    mime_type, _ = mimetypes.guess_type(file_path)

    metadata = {

        "File_Size_Bytes": stat.st_size,

        "File_Size_MB": round(stat.st_size / (1024 * 1024), 2),

        "Last_Modified":

            datetime.fromtimestamp(

                stat.st_mtime

            ).strftime("%Y-%m-%d %H:%M:%S"),

        "MIME_Type":

            mime_type if mime_type else "Unknown",

        "Readable":

            os.access(file_path, os.R_OK)

    }

    return metadata

In [ ]:
# ==========================================================
# Metadata Collection
# ==========================================================

metadata_records = []

for file in documents:

    metadata = extract_metadata(file)

    metadata_records.append(metadata)

metadata_df = pd.DataFrame(metadata_records)

metadata_df.head()

In [ ]:
# ==========================================================
# Merge Metadata
# ==========================================================

inventory_df = pd.concat(

    [

        inventory_df,

        metadata_df

    ],

    axis=1

)

inventory_df.head()

In [ ]:
# ==========================================================
# Enterprise Department Mapping
# ==========================================================

department_mapping = {

    # Engineering Manuals
    "abb": "Engineering",
    "siemens": "Engineering",
    "schneider": "Engineering",
    "atlas_copco": "Engineering",

    # Maintenance
    "maintenance": "Maintenance",

    # Safety
    "safety_and_regulations": "Safety",

    # Operations / OCR
    "equipment_labels": "Operations",
    "gauges": "Operations",
    "inspection_forms": "Quality"

}

inventory_df["Department"] = (

    inventory_df["Folder"]

    .map(department_mapping)

    .fillna("Unknown")

)

In [ ]:
# ==========================================================
# Manufacturer Extraction
# ==========================================================

manufacturer_mapping = {

    "abb": "ABB",

    "siemens": "Siemens",

    "schneider": "Schneider Electric",

    "atlas_copco": "Atlas Copco"

}

inventory_df["Manufacturer"] = (

    inventory_df["Folder"]

    .map(manufacturer_mapping)

    .fillna("Nexus Industrial")

)

In [ ]:
# ==========================================================
# Category Mapping
# ==========================================================

def classify_document(row):

    folder = row["Folder"]

    extension = row["Extension"]

    if extension==".pdf":

        if folder=="manuals":

            return "OEM Manual"

        elif folder=="maintenance":

            return "Maintenance Document"

        elif folder=="safety_and_regulations":

            return "Safety SOP"

        else:

            return "PDF Document"

    elif extension in [".png",".jpg",".jpeg"]:

        return "OCR Image"

    elif extension==".csv":

        return "Structured Dataset"

    elif extension==".txt":

        return "Text Document"

    return "Other"

inventory_df["Category"] = inventory_df.apply(

    classify_document,

    axis=1

)

In [ ]:
# ==========================================================
# Metadata Preview
# ==========================================================

display(

    inventory_df[

        [

            "Document_ID",

            "File_Name",

            "Department",

            "Category",

            "File_Size_MB",

            "MIME_Type"

        ]

    ].head(20)

)

In [ ]:
# ==========================================================
# Metadata Validation
# ==========================================================

assert inventory_df["Department"].notna().all()

assert inventory_df["Category"].notna().all()

assert inventory_df["File_Size_MB"].notna().all()

assert inventory_df["Readable"].notna().all()

print("Metadata Validation Passed")

In [ ]:
# ==========================================================
# Metadata Statistics
# ==========================================================

print("="*60)

print("Metadata Statistics")

print("="*60)

print(

    inventory_df[

        [

            "Department",

            "Category"

        ]

    ].value_counts()

)

In [ ]:
logging.info("Metadata Extraction Completed")

print()

print("="*60)

print("MILESTONE 3 COMPLETED")

print("="*60)

# Stage 3 — Enterprise File Fingerprinting

## Objective

Every enterprise document should have a unique cryptographic fingerprint.

This stage generates SHA-256 hashes for every document to:

- Detect duplicate files
- Verify file integrity
- Support incremental data ingestion
- Enable reproducible pipelines

Output

Document Hash Table

In [ ]:
# ==========================================================
# SHA256 Generator
# ==========================================================

def generate_sha256(file_path: Path, chunk_size: int = 8192) -> str:
    """
    Generate SHA-256 hash for a file.

    Parameters
    ----------
    file_path : Path
        Path to the file.

    chunk_size : int
        Bytes read per iteration.

    Returns
    -------
    str
        SHA-256 hexadecimal digest.
    """

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()

In [ ]:
# ==========================================================
# Generate Hashes
# ==========================================================

print("Generating SHA-256 hashes...")

inventory_df["SHA256"] = inventory_df["Absolute_Path"].apply(
    lambda x: generate_sha256(Path(x))
)

print("Hash generation completed.")

In [ ]:
# ==========================================================
# Preview Hashes
# ==========================================================

display(

    inventory_df[

        [

            "Document_ID",

            "File_Name",

            "SHA256"

        ]

    ].head(10)

)

In [ ]:
# ==========================================================
# Duplicate Detection
# ==========================================================

inventory_df["Duplicate"] = inventory_df.duplicated(
    subset=["SHA256"],
    keep=False
)

In [ ]:
# ==========================================================
# Duplicate Summary
# ==========================================================

duplicate_count = inventory_df["Duplicate"].sum()

print("=" * 60)
print("Duplicate Detection Summary")
print("=" * 60)

print(f"Duplicate Files : {duplicate_count}")

In [ ]:
# ==========================================================
# Show Duplicates
# ==========================================================

duplicate_df = inventory_df[
    inventory_df["Duplicate"] == True
]

if duplicate_df.empty:

    print("No duplicate files detected.")

else:

    display(

        duplicate_df[

            [

                "Document_ID",

                "File_Name",

                "SHA256"

            ]

        ]

    )

In [ ]:
# ==========================================================
# Hash Validation
# ==========================================================

assert inventory_df["SHA256"].notna().all()

assert inventory_df["SHA256"].str.len().eq(64).all()

print("SHA256 Validation Passed")

In [ ]:
# ==========================================================
# Export Hash Table
# ==========================================================

hash_table = inventory_df[
    [

        "Document_ID",

        "File_Name",

        "SHA256"

    ]

]

hash_output = INVENTORY_DIR / "file_hashes.csv"

hash_table.to_csv(

    hash_output,

    index=False

)

print(f"Hash table saved to:\n{hash_output}")

In [ ]:
# ==========================================================
# Stage Completion
# ==========================================================

logging.info("SHA256 Hash Generation Completed")

print()
print("=" * 60)
print("MILESTONE 4 COMPLETED")
print("=" * 60)

# Stage 4 — Enterprise Inventory Generation

## Objective

Transform all discovered document information into a unified enterprise inventory.

This inventory becomes the master catalog for every downstream notebook.

Outputs

- document_inventory.csv
- folder_summary.json
- dataset_statistics.json

In [ ]:
# ==========================================================
# Final Inventory
# ==========================================================

inventory_df = inventory_df.sort_values(

    by="Document_ID"

).reset_index(drop=True)

print("Inventory Sorted")

In [ ]:
# ==========================================================
# Inventory Validation
# ==========================================================

required_columns = [

    "Document_ID",

    "File_Name",

    "Extension",

    "Folder",

    "Department",

    "Category",

    "SHA256"

]

missing = [

    col

    for col in required_columns

    if col not in inventory_df.columns

]

assert len(missing)==0, f"Missing Columns : {missing}"

print("Inventory Validation Passed")

In [ ]:
# ==========================================================
# Dataset Statistics
# ==========================================================

dataset_statistics = {

    "Total Documents":

        len(inventory_df),

    "PDF Files":

        int(

            (inventory_df["Extension"]==".pdf").sum()

        ),

    "Images":

        int(

            inventory_df["Extension"]

            .isin(

                [".png",".jpg",".jpeg"]

            ).sum()

        ),

    "CSV":

        int(

            (inventory_df["Extension"]==".csv").sum()

        ),

    "TXT":

        int(

            (inventory_df["Extension"]==".txt").sum()

        ),

    "Departments":

        int(

            inventory_df["Department"]

            .nunique()

        ),

    "Duplicate Files":

        int(

            inventory_df["Duplicate"].sum()

        ),

    "Total Size (MB)":

        round(

            inventory_df["File_Size_MB"].sum(),

            2

        ),

    "Average File Size (MB)":

        round(

            inventory_df["File_Size_MB"].mean(),

            2

        )

}

dataset_statistics

In [ ]:
# ==========================================================
# Folder Summary
# ==========================================================

folder_summary = (

    inventory_df

    .groupby(

        [

            "Department",

            "Folder"

        ]

    )

    .size()

    .reset_index(

        name="Document_Count"

    )

)

folder_summary

In [ ]:
# ==========================================================
# Category Summary
# ==========================================================

category_summary = (

    inventory_df

    .groupby(

        "Category"

    )

    .size()

    .reset_index(

        name="Count"

    )

)

category_summary

In [ ]:
# ==========================================================
# Largest Files
# ==========================================================

largest_files = (

    inventory_df

    .sort_values(

        "File_Size_MB",

        ascending=False

    )

    .head(10)

)

largest_files

In [ ]:
# ==========================================================
# Statistics Report
# ==========================================================

print("="*70)

print("ENTERPRISE DATASET SUMMARY")

print("="*70)

for key,value in dataset_statistics.items():

    print(f"{key:<30}: {value}")

print("="*70)

In [ ]:
# ==========================================================
# Export Inventory
# ==========================================================

inventory_output = (

    INVENTORY_DIR/

    "document_inventory.csv"

)

inventory_df.to_csv(

    inventory_output,

    index=False

)

print(inventory_output)

In [ ]:
# ==========================================================
# Export Folder Summary
# ==========================================================

folder_output = (

    INVENTORY_DIR/

    "folder_summary.json"

)

folder_summary.to_json(

    folder_output,

    orient="records",

    indent=4

)

print(folder_output)

In [ ]:
# ==========================================================
# Export Statistics
# ==========================================================

statistics_output = (

    INVENTORY_DIR/

    "dataset_statistics.json"

)

with open(

    statistics_output,

    "w"

) as f:

    json.dump(

        dataset_statistics,

        f,

        indent=4

    )

print(statistics_output)

In [ ]:
# ==========================================================
# Verify Exports
# ==========================================================

outputs = [

    inventory_output,

    hash_output,

    folder_output,

    statistics_output

]

for file in outputs:

    print(

        file.name,

        ":",

        file.exists()

    )

In [ ]:
# ==========================================================
# Final Preview
# ==========================================================

display(

    inventory_df.head(20)

)

In [ ]:
# ==========================================================
# Final Assertions
# ==========================================================

assert inventory_output.exists()

assert hash_output.exists()

assert folder_output.exists()

assert statistics_output.exists()

print("All Outputs Generated Successfully")

In [ ]:
# ==========================================================
# Notebook Completion
# ==========================================================

logging.info("Notebook 01 Completed Successfully")

print()

print("="*80)

print("NOTEBOOK 01 COMPLETED")

print("="*80)

In [ ]:
# ==========================================================
# Next Notebook
# ==========================================================

print("""

Notebook 02

Enterprise Document Processing Pipeline

Input

document_inventory.csv

Output

processed_documents.parquet

ocr_queue.csv

text_corpus.parquet

""")

In [ ]:
# ==========================================================
# Verify Departments
# ==========================================================

print("=" * 60)
print("ENTERPRISE DEPARTMENTS")
print("=" * 60)

departments = sorted(inventory_df["Department"].unique())

print(f"Total Departments : {len(departments)}\n")

for i, dept in enumerate(departments, start=1):
    count = (inventory_df["Department"] == dept).sum()
    print(f"{i}. {dept:<15} -> {count} documents")

In [ ]:
# ==========================================================
# One Example Document From Each Department
# ==========================================================

examples = (
    inventory_df
    .groupby("Department")
    .first()
    .reset_index()
)

display(
    examples[
        [
            "Department",
            "File_Name",
            "Category",
            "Folder",
            "Extension"
        ]
    ]
)

In [ ]:
# ==========================================================
# Category Distribution
# ==========================================================

category_summary = (
    inventory_df
    .groupby("Category")
    .size()
    .reset_index(name="Count")
    .sort_values("Count", ascending=False)
)

display(category_summary)

In [ ]:
# ==========================================================
# Folder Verification
# ==========================================================

folder_summary = (
    inventory_df
    .groupby(["Department", "Folder"])
    .size()
    .reset_index(name="Documents")
)

display(folder_summary)

In [ ]:
# ==========================================================
# Enterprise Inventory Overview
# ==========================================================

print("=" * 90)
print("ENTERPRISE INVENTORY")
print("=" * 90)

display(
    inventory_df[
        [
            "Document_ID",
            "Department",
            "Category",
            "File_Name",
            "Extension",
            "File_Size_MB",
            "Duplicate"
        ]
    ]
)